# పాఠం 13 - కోగ్నీ నాలెడ్జ్ గ్రాఫ్‌లతో ఏజెంట్ మెమరీ


## సెటప్

ఈ నోట్‌బుక్ స్థిరమైన జ్ఞాపకశక్తితో కూడిన ఒక తెలివైన **కోడ్ సహాయకుడు**ను [**Cognee**](https://www.cognee.ai/) నోలెడ్జ్ గ్రాఫ్లు మరియు **Microsoft Agent Framework** (MAF) ఉపయోగించి ఎలా నిర్మించాలో చూపిస్తుంది.

Cognee అనిర్దిష్ట పాఠ్యాన్ని వెక్టర్ ఎంబెడ్డింగ్‌ల ద్వారా మద్దతుగా ఉండే, స్ట్రక్చర్డ్, క్వెరీ చేయదగిన నోలెడ్జ్ గ్రాఫ్‌గా మార్చి — మీ ఏజెంట్‌కు సంపన్న, సంబంధ-జ్ఞానంతో కూడిన దీర్ఘకాలిక జ్ఞాపకశక్తిని అందిస్తుంది.

### మీరు నేర్చుకునేది ఏంటి
1. **నోలెడ్జ్ గ్రాఫ్స్ నిర్మాణం**: డెవలపర్ ప్రొఫైల్‌లు మరియు ఉత్తమ పద్ధతులను స్ట్రక్చర్డ్, క్వెరీ చేయదగిన జ్ఞానంగా మార్చడం.
2. **Cogneeని MAF తో సమ్మేళనం చేయడం**: `@tool` ఫంక్షన్‌లను ఉపయోగించి MAF ఏజెంట్ Cognee యొక్క నోలెడ్జ్ గ్రాఫ్‌ను క్వెరీ చేయడానికి వీలు కల్పించడం.
3. **సెషన్-అవేర్ సంభాషణలు**: ఒకే సెషన్‌లోని ఒకటి కంటే ఎక్కువ ప్రశ్నలపై సందర్భం కొనసాగించడం.
4. **దీర్ఘకాలిక జ్ఞాపకశక్తి**: ముఖ్యమైన జ్ఞానాన్ని సెషన్‌ల మీద నిల్వ చేసి, కొత్త సంభాషణలలో తీసుకురావడం.

### ముందస్తు అవసరాలు
- Python 3.9+
- స్థానికంగా Redis నడుస్తోంది (`docker run -d -p 6379:6379 redis`) సెషన్ నిర్వహణ కోసం
- LLM API కీ (ఉదాహరణకు OpenAI) — `.env`లో `LLM_API_KEY` సెట్ చేయాలి
- `.env` లో `CACHING=true` (Cognee సెషన్లకు అవసరం)
- ఒక Azure AI Foundry ప్రాజెక్టు, సవరించిన చాట్ మోడల్‌తో
- `.env`లో `AZURE_AI_PROJECT_ENDPOINT` మరియు `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI లో లాగిన్ అయి ఉండాలి (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ఏజెంట్ గుర్తుకుళ్ల రకాలు

ఈ నోటుబుక్ మెయిన్ లెసన్ 13 నోటుబుక్ నుండి అదే మూడు గుర్తుకుళ్ల రకాలను అన్వేషిస్తుంది, కానీ దీర్ఘకాలిక గుర్తుకు Cognee ను బ్యాక్ ఎండ్ గా ఉపయోగిస్తుంది:

| గుర్తుకుళ్ల రకం | Mexhanism | కాలం |
|---|---|---|
| **వర్కింగ్** | `agent.create_session()` (MAF) | ఒక్క సంభాషణ |
| **సంఖ్యా-కాలం** | Cognee సెషన్ కాష్ (Redis) | ఒక్క సెషన్ |
| **దీర్ఘకాలం** | Cognee జ్ఞాన గ్రాఫ్ + వెక్టర్లు | శాశ్వతం |

### Cognee యొక్క గుర్తుకు నిర్మాణం
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## కోగ్నీ స్టోరేజ్ సిద్ధం చేయండి


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## భాగం 1 — జ్ఞాన బేస్ నిర్మాణం

మన కోడింగ్ సహాయకుడికి సమగ్ర జ్ఞాన బేస్ సృష్టించేందుకు మేము మూడు రకాల డేటాను తీసుకుంటాము:

1. **డెవలపర్ ప్రొఫైల్** — వ్యక్తిగత నైపుణ్యం మరియు సాంకేతిక నేపథ్యం  
2. **పైథాన్ బెస్ట్ ప్రాక్టీసెస్** — ప్రాథమిక మార్గదర్శకాలతో పైథాన్ జెన్  
3. **చరిత్రాత్మక సంభాషణలు** — డెవలపర్లు మరియు AI సహాయకుల మధ్య గత Q&A సెషన్లు


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## జ్ఞాన గ్రాఫ్‌ను దర్శించండి

Cognee తనకు అమర్చిన ఇన్టరాక్టివ్ HTML విజువలైజేషన్ ద్వారా ఎంటిటీలు మరియు సంబంధాలను ప్రదర్శించగలదు.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify తో మెమరీ ని సమృద్ధిచేయండి

`memify()` జ్ఞాన గ్రాఫ్‌ను విశ్లేషించి, తెలివైన నియమాలను సృష్టిస్తుంది — నమూనాలు, ఉత్తమ ప్రవర్తనలు, మరియు సాంధర్భాల మధ్య సంబంధాలను గుర్తిస్తుంది.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — Cognee Tools తో MAF ఏజెంట్

ఇప్పుడు మేము `@tool` ఫంక్షన్ల ద్వారా Cognee యొక్క జ్ఞాన గ్రాఫ్‌ను క్వెరి చేయగల MAF ఏజెంట్‌ను సృష్టిస్తున్నాము. ఇది ఏజెంట్‌కు సెషన్ల ద్వారా సంభాషణా సందర్భాన్ని నిర్వహిస్తూ గ్రాఫ్-ప్రజ్ఞాపూర్వక సెమాంటిక్ శోధన యొక్క పూర్తి శక్తిని ఉపయోగించడానికి అనుమతిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## సెషన్లతో పని చేసే మెమొరీ

`AgentSession` (`agent.create_session()` ద్వారా సృష్టించబడుతుంది) సెషన్‌లో పని చేసే మెమొరీని అందిస్తుంది. ఏజెంట్ మొదటి సందేశాలను మళ్ళీ చూచి, అలాగే Cognee యొక్క దీర్ఘకాలిక జ్ఞానం గ్రాఫ్‌ను కూడా విచారించ గలదు.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## కొత్త సెషన్ — దీర్ఘకాలిక జ్ఞాపకం కొనసాగుతుంది

కొత్త సెషన్ ప్రారంభించడం వర్కింగ్ మెమరీని క్లియర్ చేస్తుంది, కానీ Cognee నాలెడ్జ్ గ్రాఫ్ ఇంకా అందుబాటులో ఉంటుంది. ఏజెంట్ పూర్తిగా కొత్త సంభాషణలో అదే దీర్ఘకాలిక జ్ఞానాన్ని తెప్పించవచ్చు.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## సమారాంశం

ఈ నోట్‌బుక్‌లో మీరు **MAF యొక్క వర్కింగ్ మెమోరీ** (`agent.create_session()`) మరియు **Cognee యొక్క దీర్ఘకాలిక జ్ఞాన గ్రాఫ్** ని సంయోజించే కోడింగ్ అసిస్టెంట్‌ను నిర్మించారు.

### మీరు నేర్చుకున్నది
1. **జ్ఞాన గ్రాఫ్ నిర్మాణం**: Cognee నిర్మాణ రహిత వచనం ని తీసుకొని గ్రాఫ్ + వెక్టర్ మెమరీని తయారుచేస్తుంది.
2. **memify తో గ్రాఫ్ శ్రద్ధ**: ఇప్పటికే ఉన్న గ్రాఫ్ పై ఆధారంగా ఫాక్ట్లు మరియు సమృద్ధిగా సంబంధాలు తయారుచేస్తుంది.
3. **MAF + Cognee సంయోగం**: `@tool` ఫంక్షన్లు MAF ఏజెంట్ Cognee గ్రాఫ్ ను సహజంగా క్వెరీ చేయేటట్టు చేస్తాయి.
4. **వర్కింగ్ మెమరీ + దీర్ఘకాలిక మెమరీ**: `AgentSession` (`agent.create_session()` ద్వారా) సెషన్ సందర్భాన్ని అందిస్తుంటే Cognee నిరంతర జ్ఞానాన్ని అందిస్తుంది.
5. **NodeSetsతో ఫిల్టర్ చేసిన సెర్చ్**: జ్ఞాన గ్రాఫ్ యొక్క నిర్దిష్ట ఉపసెట్టలను లక్ష్యం గా చేసుకోవచ్చు (ఉదా: కేవలం సూత్రాలు).

### ముఖ్య సారాంశాలు
- **Cognee** రా వచనం ని నిర్మాణాత్మక, సంబంధ-అవగాహన కలిగిన మెమరీ గా మార్చుతుంది — ఇది ఒక సాధారణ వెక్టర్ స్టోర్ కన్నా బలవంతంగా ఉంటుంది.
- **`@tool` ఫంక్షన్లు** MAF ఏజెంట్లను బాహ్య జ్ఞాన వ్యవస్థలతో స్పష్టంగా జతచేస్తాయి.
- **`AgentSession`** (`agent.create_session()` ద్వారా) ప్రతి సంభాషణ సందర్భాన్ని దీర్ఘకాలిక జ్ఞానంతో వేరుగా ఉంచుతుంది.
- యే జ్ఞాన గ్రాఫ్ అనేక సెషన్లు మరియు ఏజెంట్లకు సేవలందిస్తుంది.

### వాస్తవ ప్రపంచ అనువర్తనాలు
- **డెవలపర్ కోపైలాట్లు**: కోడ్ సమీక్ష, ఘటన విశ్లేషణ, వ్యవస్థాపన సహాయకులు
- **గ్రాహక-ముఖి కోపైలాట్లు**: ఉత్పత్తి డాక్యుమెంట్ల, FAQs, CRM గమనికల మీద మద్దతు ఏజెంట్లు
- **అంతర్గత నిపుణుల కోపైలాట్లు**: విధాన, చట్ట, లేదా భద్రత సహాయకులు మార్గదర్శకాలు విశ్లేషణ కోసం
- **ఒక్కటీ చేయబడిన డేటా లేయర్స్**: నిర్మాణాత్మక మరియు నిర్మాణ రహిత డేటాను ఒక ప్రశ్నించదగిన గ్రాఫ్‌లో కలపడం

### తరువాతి దశలు
- Cognee లో కాల సంబంధ అవగాహనతో ప్రయోగాలు చేయండి
- డొమైన్-నిర్దిష్ట గ్రాఫ్ నాణ్యత కోసం OWL ఆంటాలజీ నిర్వచించండి
- సమయంతో మెరుగుపరచడానికి వినియోగదారు అభిప్రాయం లూపులను జోడించండి
- ఒకే Cognee మెమరీ లేయర్‌ను పంచుకునే బహుళ ఏజెంట్ వ్యవస్థలకు స్కేల్ చేయండి


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్పష్టత:**
ఈ డాక్యుమెంట్‌ను AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడ్డది. మనం ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలలో పొరపాట్లు లేదా లోపాలు ఉండవచ్చని దయచేసి గమనించండి. మੂల డాక్యుమెంట్ దాని స్థానిక భాషలోనే అధికారం ఉన్న మూలంగా పరిగణించాలి. ముఖ్యమైన సమాచారానికి, ప్రొఫెషనల్ మానవ అనువాదం చేయించుకోవడం మేలైనది. ఈ అనువాదం వాడకం వల్ల జరిగిన ఏమైనా అవగాహన లోపాలు లేదా తప్పుదారుల గురించి మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
